# TRUJAMAn 🧙 · 2. Test

**T**extual **R**ecognition **U**nit for **J**udeo-**A**rabic (and Hebrew) in **M**edieval **A**nthologies · *beta v0.1.0*

This notebook tests the **published model** (`trujaman_v0.1-beta.joblib`) on the 40 works held out in `1.Data/b.Training/b.2.Evaluation/`. It does not retrain anything. Like `1.Training.ipynb`, it needs the FJMS texts, which are not distributed with this repository.

1. Accuracy per work and per author
2. Precision and recall for Hebrew and Judeo-Arabic
3. Held-out accuracy: authors seen and unseen in training
4. Confusion matrix
5. Errors at language switches
6. Confidence
7. Most frequent words: mislabelled, Judeo-Arabic, Hebrew
8. Most frequent chunks: mislabelled, Judeo-Arabic, Hebrew

The outputs show counts and scores only, plus the ten most frequent single words and chunks: never running text.

In [1]:
# ===========================================================================
EVAL_DIR   = "1.Data/b.Training/b.2.Evaluation"   # held-out works
TRAIN_DIR  = "1.Data/b.Training/b.1.Training"     # only its works.csv is read
MODEL_PATH = "trujaman_v0.1-beta.joblib"
# ===========================================================================

import re, glob, os
from collections import Counter
import numpy as np, pandas as pd, joblib
from sklearn.metrics import (accuracy_score, classification_report, confusion_matrix,
                             f1_score, precision_score, recall_score, roc_auc_score)

pd.set_option("display.width", 140)

# Same cleaning and features as 1.Training.ipynb: change both or neither.
# 05C4 upper dot | 05BF rafe | 05F3/05F4 geresh, gershayim | ' " | punctuation
STRIP = re.compile("[\u05c4\u05bf\u05f3\u05f4\u0027\u0022,.;:()\\[\\]\\-\u2013\u2014?!\u2026]")


def normalise(s):
    return STRIP.sub("", str(s))


def load_dir(folder):
    """Read every work CSV in a folder into one frame, tagged by source file."""
    paths = sorted(p for p in glob.glob(os.path.join(folder, "*.csv"))
                   if os.path.basename(p) not in ("works.csv", "evaluation_scores.csv"))
    if not paths:
        raise FileNotFoundError(
            f"No texts found in {folder}. The FJMS texts are not distributed with "
            "this repository: see 1.Training.ipynb on how to add them.")
    frames = []
    for p in paths:
        d = pd.read_csv(p)
        d = d[d.lang.isin(["JA", "HE"])].copy()
        d["work"] = os.path.splitext(os.path.basename(p))[0]
        d["doc"] = d.work + ":" + d.wID.astype(str).str.split(".").str[0]
        d["w"] = d.word.map(normalise)
        frames.append(d[d.w.str.len() > 0])
    return pd.concat(frames, ignore_index=True)


def features(d):
    """Target word marked with ~, flanked by its neighbours on the same page."""
    prev = d.w.groupby(d.doc).shift(1).fillna("")
    nxt = d.w.groupby(d.doc).shift(-1).fillna("")
    return (prev + " ~" + d.w + "~ " + nxt).values


def scores(x):
    """Accuracy, precision, recall and F1 for one group of words."""
    y, p = (x.lang == "HE").astype(int), (x.pred == "HE").astype(int)
    return pd.Series({
        "works": x.work.nunique() if "work" in x else 1, "words": len(x), "pct_HE": y.mean(),
        "accuracy": accuracy_score(y, p),
        "P_HE": precision_score(y, p, zero_division=0), "R_HE": recall_score(y, p, zero_division=0),
        "F1_HE": f1_score(y, p, zero_division=0),
        "P_JA": precision_score(y, p, pos_label=0, zero_division=0),
        "R_JA": recall_score(y, p, pos_label=0, zero_division=0),
        "F1_JA": f1_score(y, p, pos_label=0, zero_division=0)})


clf = joblib.load(MODEL_PATH)
ev = load_dir(EVAL_DIR)
ev["p_HE"] = clf.predict_proba(features(ev))[:, 1]
ev["pred"] = np.where(ev.p_HE > 0.5, "HE", "JA")
ev["correct"] = ev.lang == ev.pred

# Authors: an evaluation author is "seen" if any of their works is in training.
ev_works = pd.read_csv(os.path.join(EVAL_DIR, "works.csv"))
tr_works = pd.read_csv(os.path.join(TRAIN_DIR, "works.csv"))
ev["author"] = ev.work.map(ev_works.set_index("codeName").author)
seen_authors = set(tr_works.author) - {"Unknown"}
ev["authors"] = np.select([ev.author == "Unknown", ev.author.isin(seen_authors)],
                          ["unknown", "seen"], "unseen")

print(f"{len(ev):,} words in {ev.work.nunique()} works by {ev.author.nunique()} authors; "
      f"{(ev.lang == 'HE').mean():.1%} Hebrew")

1,070,917 words in 40 works by 34 authors; 36.2% Hebrew


## 1. Accuracy per work and per author

Worst first. *Pooled* accuracy counts every word equally, so large works weigh more; the *mean over works* counts every work equally.

In [2]:
per_work = (ev.groupby("work")
              .apply(lambda x: pd.Series({"author": x.author.iat[0], "authors": x.authors.iat[0],
                                          "words": len(x), "pct_HE": (x.lang == "HE").mean(),
                                          "accuracy": x.correct.mean()}), include_groups=False)
              .sort_values("accuracy"))
print(per_work.to_string(formatters={"pct_HE": "{:.1%}".format, "accuracy": "{:.1%}".format,
                                     "words": "{:,}".format}))

per_author = (ev.groupby("author")
                .apply(lambda x: pd.Series({"works": x.work.nunique(), "words": len(x),
                                            "accuracy": x.correct.mean()}), include_groups=False)
                .sort_values("accuracy")
                .astype({"works": int, "words": int}))
print("\nPER AUTHOR")
print(per_author.to_string(formatters={"accuracy": "{:.1%}".format, "words": "{:,}".format}))

print(f"\npooled accuracy         {ev.correct.mean():.1%}")
print(f"mean accuracy over works {per_work.accuracy.mean():.1%}")

                                                author  authors   words pct_HE accuracy
work                                                                                   
IbnQuraysh_Aygrt                     Judah ibn Quraysh   unseen  18,034  45.5%    89.4%
SaadiaG_rvt                                Saadia Gaon     seen   1,376   6.5%    91.6%
IbnBarun_spr                           Isaac ibn Barun   unseen  22,941  21.8%    92.6%
Hayyuj_spr                                Judah Hayyuj   unseen  17,356  40.1%    93.4%
alHarizi_ms`y                          Judah al-Harizi     seen   3,339  64.7%    93.4%
NathanYesh_prvSH                  Nathan Av ha-Yeshiva   unseen  57,872  41.5%    93.6%
TanhumYeru_dnyAl                  Tanhum ha-Yerushalmi     seen  10,328  24.9%    93.7%
BenHofniG_mbvA                   Samuel ben Hofni Gaon   unseen  18,532  64.8%    94.1%
DavidBenAbraMaim_brASHyt  David ben Abraham Maimonides     seen   9,016  32.9%    94.8%
IbnGhiyath_khlt                 

## 2. Precision and recall for Hebrew and Judeo-Arabic

*Precision*: of the words labelled HE (or JA), how many really are. *Recall*: of the words that really are HE (or JA), how many were found. In works with under 1% Hebrew, a few errors weigh heavily on the Hebrew scores; those works are marked with `*`.

In [3]:
y, p = (ev.lang == "HE").astype(int), (ev.pred == "HE").astype(int)
print("ALL EVALUATION WORDS")
print(classification_report(y, p, target_names=["JA", "HE"], digits=3))

pr = ev.groupby("work").apply(scores, include_groups=False).sort_values("F1_HE")
pr.index = [w + (" *" if s < 0.01 else "") for w, s in zip(pr.index, pr.pct_HE)]
cols = ["words", "pct_HE", "P_HE", "R_HE", "F1_HE", "P_JA", "R_JA", "F1_JA"]
print(pr[cols].to_string(formatters={"words": "{:,.0f}".format, "pct_HE": "{:.1%}".format,
                                     **{c: "{:.3f}".format for c in cols[2:]}}))

ALL EVALUATION WORDS


              precision    recall  f1-score   support

          JA      0.980     0.964     0.972    683038
          HE      0.938     0.965     0.951    387879

    accuracy                          0.964   1070917
   macro avg      0.959     0.964     0.962   1070917
weighted avg      0.965     0.964     0.964   1070917



                           words pct_HE  P_HE  R_HE F1_HE  P_JA  R_JA F1_JA
Maim_bAvr *                6,709   0.1% 0.034 0.400 0.063 1.000 0.992 0.996
alMuqammis_`SHrym *       26,428   0.4% 0.139 0.747 0.235 0.999 0.983 0.991
SaadiaG_rvt                1,376   6.5% 0.410 0.663 0.506 0.976 0.934 0.954
IbnBarun_spr              22,941  21.8% 0.800 0.879 0.837 0.965 0.939 0.952
JudahHaLev_spr            47,351  11.4% 0.817 0.893 0.853 0.986 0.974 0.980
TanhumYeru_dnyAl          10,328  24.9% 0.860 0.894 0.877 0.964 0.952 0.958
IbnGhiyath_khlt           33,013  18.2% 0.809 0.961 0.879 0.991 0.950 0.970
IbnEzra_h`yvnym           36,237  16.1% 0.814 0.964 0.882 0.993 0.958 0.975
IbnQuraysh_Aygrt          18,034  45.5% 0.858 0.920 0.888 0.929 0.873 0.900
BenHofniG_`SHr             9,794   7.1% 0.842 0.960 0.897 0.997 0.986 0.991
IbnAknin_htglvt           82,297  20.3% 0.869 0.960 0.912 0.989 0.963 0.976
Hayyuj_spr                17,356  40.1% 0.936 0.897 0.916 0.933 0.959 0.946
MubashshirLe

## 3. Held-out accuracy: authors seen and unseen in training

Every evaluation work is held out. The question here is whether the model also generalises to **authors** it has never seen. Authors with a single work in the corpus appear only in evaluation, and Ben Hofni was kept out of training on purpose. The anonymous work is counted separately, as "Unknown" is not one author.

In [4]:
groups = {"authors unseen in training": ev.authors == "unseen",
          "  of which Ben Hofni": ev.author == "Samuel ben Hofni Gaon",
          "authors also in training": ev.authors == "seen",
          "unknown author": ev.authors == "unknown",
          "all": ev.authors.notna()}
held = pd.DataFrame({name: scores(ev[m]) for name, m in groups.items()}).T
held.insert(1, "authors", [ev[m].author.nunique() for m in groups.values()])
held.insert(4, "share_of_words", held.words / len(ev))
held["AUC"] = [roc_auc_score(ev[m].lang == "HE", ev[m].p_HE) for m in groups.values()]
cols = ["works", "authors", "words", "share_of_words", "accuracy", "AUC", "F1_JA", "F1_HE"]
print(held[cols].to_string(formatters={"works": "{:.0f}".format, "authors": "{:.0f}".format,
                                       "words": "{:,.0f}".format, "share_of_words": "{:.1%}".format,
                                       "accuracy": "{:.1%}".format, "AUC": "{:.3f}".format,
                                       "F1_JA": "{:.3f}".format, "F1_HE": "{:.3f}".format}))

                           works authors     words share_of_words accuracy   AUC F1_JA F1_HE
authors unseen in training    31      25   974,869          91.0%    96.4% 0.993 0.972 0.950
  of which Ben Hofni           7       1   144,472          13.5%    96.3% 0.993 0.971 0.952
authors also in training       8       8    95,493           8.9%    96.6% 0.994 0.968 0.964
unknown author                 1       1       555           0.1%    95.3% 0.993 0.964 0.932
all                           40      34 1,070,917         100.0%    96.4% 0.994 0.972 0.951


## 4. Confusion matrix

Rows are the true language, columns the model's label. Percentages are per row: the share of each language that was labelled correctly or not.

In [5]:
cm = pd.DataFrame(confusion_matrix(ev.lang, ev.pred, labels=["HE", "JA"]),
                  index=["true HE", "true JA"], columns=["labelled HE", "labelled JA"])
pct = cm.div(cm.sum(axis=1), axis=0)
print("WORDS")
print(cm.map("{:,}".format).to_string())
print("\nSHARE OF EACH TRUE LANGUAGE")
print(pct.map("{:.1%}".format).to_string())
print(f"\nHebrew labelled Hebrew:   {cm.iat[0, 0]:>9,}")
print(f"Judeo-Arabic labelled JA: {cm.iat[1, 1]:>9,}")
print(f"Judeo-Arabic labelled HE: {cm.iat[1, 0]:>9,}")
print(f"Hebrew labelled JA:       {cm.iat[0, 1]:>9,}")

WORDS
        labelled HE labelled JA
true HE     374,330      13,549
true JA      24,760     658,278

SHARE OF EACH TRUE LANGUAGE
        labelled HE labelled JA
true HE       96.5%        3.5%
true JA        3.6%       96.4%

Hebrew labelled Hebrew:     374,330
Judeo-Arabic labelled JA:   658,278
Judeo-Arabic labelled HE:    24,760
Hebrew labelled JA:          13,549


## 5. Errors at language switches

A word is *at a switch* if the word before or after it on the same page is in the other language. A *run* is a stretch of consecutive words in the same language; short runs are, for example, a Hebrew term inside Arabic prose.

In [6]:
prev = ev.groupby("doc").lang.shift(1)
nxt = ev.groupby("doc").lang.shift(-1)
ev["at_switch"] = (prev.notna() & (prev != ev.lang)) | (nxt.notna() & (nxt != ev.lang))

sw = ev.groupby("at_switch").agg(words=("correct", "size"), error_rate=("correct", lambda c: 1 - c.mean()))
sw.index = sw.index.map({True: "at a switch", False: "inside a run"})
sw.insert(1, "share_of_words", sw.words / len(ev))
sw["share_of_errors"] = ev.groupby("at_switch").correct.apply(lambda c: (~c).sum()).values / (~ev.correct).sum()
print(sw.to_string(formatters={"words": "{:,}".format, "share_of_words": "{:.1%}".format,
                               "error_rate": "{:.1%}".format, "share_of_errors": "{:.1%}".format}))

# length of the run each word belongs to
run_id = ((ev.lang != prev) | (ev.doc != ev.doc.shift(1))).cumsum()
ev["run_len"] = ev.groupby(run_id).lang.transform("size")
ev["run"] = pd.cut(ev.run_len, [0, 1, 2, 5, 20, np.inf], labels=["1 word", "2", "3-5", "6-20", "over 20"])
runs = ev.groupby(["run", "lang"], observed=True).agg(words=("correct", "size"),
                                                      error_rate=("correct", lambda c: 1 - c.mean()))
print("\nERROR RATE BY LENGTH OF RUN")
print(runs.unstack("lang").to_string(formatters={("words", "HE"): "{:,}".format, ("words", "JA"): "{:,}".format,
                                                 ("error_rate", "HE"): "{:.1%}".format,
                                                 ("error_rate", "JA"): "{:.1%}".format}))

               words share_of_words error_rate share_of_errors
at_switch                                                     
inside a run 894,395          83.5%       2.0%           47.5%
at a switch  176,522          16.5%      11.4%           52.5%

ERROR RATE BY LENGTH OF RUN
          words         error_rate      
lang         HE      JA         HE    JA
run                                     
1 word   12,477   8,965      27.4% 27.3%
2        20,840  14,322       8.3% 11.6%
3-5      53,268  45,020       5.5%  6.8%
6-20    109,816 178,241       2.6%  4.0%
over 20 191,478 436,490       1.4%  2.4%


## 6. Confidence

The model gives every word a probability of being Hebrew. Words near 0.5 are the ones it is unsure about. This shows how often it is wrong at each level of confidence.

In [7]:
bands = [0, 0.05, 0.2, 0.4, 0.6, 0.8, 0.95, 1.0001]
names = ["0-0.05 (sure JA)", "0.05-0.2", "0.2-0.4", "0.4-0.6 (unsure)", "0.6-0.8", "0.8-0.95", "0.95-1 (sure HE)"]
ev["band"] = pd.cut(ev.p_HE, bands, labels=names, right=False)
conf = ev.groupby("band", observed=True).agg(words=("correct", "size"),
                                             error_rate=("correct", lambda c: 1 - c.mean()))
conf.insert(1, "share_of_words", conf.words / len(ev))
print(conf.to_string(formatters={"words": "{:,}".format, "share_of_words": "{:.1%}".format,
                                 "error_rate": "{:.1%}".format}))

unsure = ev.p_HE.between(0.2, 0.8)
print(f"\nunsure words (0.2-0.8): {unsure.mean():.1%} of all words, wrong {1 - ev.correct[unsure].mean():.1%} of the time")
print(f"all other words:        {(~unsure).mean():.1%} of all words, wrong {1 - ev.correct[~unsure].mean():.1%} of the time")
print(f"share of all errors among unsure words: {(~ev.correct[unsure]).sum() / (~ev.correct).sum():.1%}")

                   words share_of_words error_rate
band                                              
0-0.05 (sure JA) 545,397          50.9%       0.2%
0.05-0.2          81,209           7.6%       4.0%
0.2-0.4           33,626           3.1%      16.0%
0.4-0.6 (unsure)  22,357           2.1%      43.2%
0.6-0.8           27,291           2.5%      36.1%
0.8-0.95          59,380           5.5%      10.4%
0.95-1 (sure HE) 301,657          28.2%       0.9%

unsure words (0.2-0.8): 7.8% of all words, wrong 29.9% of the time
all other words:        92.2% of all words, wrong 1.4% of the time
share of all errors among unsure words: 65.0%


## 7. Most frequent words: mislabelled, Judeo-Arabic, Hebrew

Words after cleaning (diacritics and punctuation removed). For each word: how often it occurs, and how often the model gets it wrong.

In [8]:
by_word = ev.groupby("w").agg(occurrences=("correct", "size"), errors=("correct", lambda c: (~c).sum()))
by_word["error_rate"] = by_word.errors / by_word.occurrences
fmt = {"occurrences": "{:,}".format, "errors": "{:,}".format, "error_rate": "{:.1%}".format}

print("10 MOST FREQUENTLY MISLABELLED WORDS")
print(by_word.sort_values("errors", ascending=False).head(10).to_string(formatters=fmt))

for lang, name in [("JA", "JUDEO-ARABIC"), ("HE", "HEBREW")]:
    x = ev[ev.lang == lang].groupby("w").agg(occurrences=("correct", "size"),
                                             errors=("correct", lambda c: (~c).sum()))
    x["error_rate"] = x.errors / x.occurrences
    print(f"\n10 MOST FREQUENT {name} WORDS")
    print(x.sort_values("occurrences", ascending=False).head(10).to_string(formatters=fmt))

10 MOST FREQUENTLY MISLABELLED WORDS
    occurrences errors error_rate
w                                
לא       10,363    735       7.1%
מן       20,553    504       2.5%
ולא       5,831    433       7.4%
או        3,405    309       9.1%
אלא       3,918    295       7.5%
כל        4,525    246       5.4%
לה        4,518    234       5.2%
פי       22,112    193       0.9%
אל        2,238    163       7.3%
עלי       9,628    152       1.6%



10 MOST FREQUENT JUDEO-ARABIC WORDS
     occurrences errors error_rate
w                                 
פי        21,738     39       0.2%
מן        19,280    290       1.5%
אן        13,850      0       0.0%
עלי        9,387     34       0.4%
מא         8,303      3       0.0%
דלך        6,782      0       0.0%
לא         6,272    437       7.0%
אלי        6,023     86       1.4%
אלדי       4,870      0       0.0%
קאל        4,363      0       0.0%



10 MOST FREQUENT HEBREW WORDS
    occurrences errors error_rate
w                                
את        4,134      0       0.0%
לא        4,091    298       7.3%
על        3,548      2       0.1%
כל        2,650    101       3.8%
כי        2,402     14       0.6%
ולא       2,047    223      10.9%
אמ        1,890      1       0.1%
ר         1,835      0       0.0%
לו        1,819     19       1.0%
אל        1,791     75       4.2%


## 8. Most frequent chunks: mislabelled, Judeo-Arabic, Hebrew

The model does not see whole words: it sees **chunks** of 1 to 5 letters taken from the word and its neighbours, each with a learned weight. Here the chunks are taken from each word on its own, with `_` marking the start or end of the word (so `_אל` is a word beginning with אל). Single letters are left out, as they occur everywhere.

The last two tables show the chunks the model itself weights most strongly towards each language, which is the most direct view of what it has learned.

In [9]:
vec, lr = clf.steps[0][1], clf.steps[-1][1]
analyse = vec.build_analyzer()


def chunk_counts(words):
    """Count the chunks (2-5 letters) in a Series of words, weighted by occurrences."""
    counts = Counter()
    for w, n in words.value_counts().items():
        for c in set(analyse(w)):
            if len(c.strip()) >= 2:
                counts[c.replace(" ", "_")] += n
    return counts


all_chunks = chunk_counts(ev.w)
err_chunks = chunk_counts(ev.w[~ev.correct])
top_err = pd.DataFrame(err_chunks.most_common(10), columns=["chunk", "in mislabelled words"]).set_index("chunk")
top_err["in all words"] = [all_chunks[c] for c in top_err.index]
top_err["error_rate"] = top_err["in mislabelled words"] / top_err["in all words"]
print("10 MOST FREQUENT CHUNKS IN MISLABELLED WORDS")
print(top_err.to_string(formatters={"in mislabelled words": "{:,}".format, "in all words": "{:,}".format,
                                    "error_rate": "{:.1%}".format}))

he_chunks = chunk_counts(ev.w[ev.lang == "HE"])
for lang, name in [("JA", "JUDEO-ARABIC"), ("HE", "HEBREW")]:
    c = chunk_counts(ev.w[ev.lang == lang])
    top = pd.DataFrame(c.most_common(10), columns=["chunk", f"in {lang} words"]).set_index("chunk")
    top["share_HE"] = [he_chunks[k] / all_chunks[k] for k in top.index]
    print(f"\n10 MOST FREQUENT CHUNKS IN {name} WORDS  (share_HE: how often the chunk occurs in Hebrew words)")
    print(top.to_string(formatters={f"in {lang} words": "{:,}".format, "share_HE": "{:.1%}".format}))

# strongest chunks according to the model (target-word chunks, marked with ~)
names, coef = vec.get_feature_names_out(), lr.coef_[0]
order = np.argsort(coef)


def strongest(idx, n=10):
    """First n target-word chunks in idx, counting padded variants (' ~אל', '~אל') once."""
    out, seen = [], set()
    for i in idx:
        key = names[i].strip()
        if "~" in key and key not in seen:
            seen.add(key)
            out.append((key, round(coef[i], 2)))
        if len(out) == n:
            break
    return pd.DataFrame(out, columns=["chunk", "weight"])


for label, idx in [("JUDEO-ARABIC", order), ("HEBREW", order[::-1])]:
    print(f"\n10 CHUNKS WEIGHTED MOST STRONGLY TOWARDS {label}  (~ marks the edge of the word being labelled)")
    print(strongest(idx).to_string(index=False))

10 MOST FREQUENT CHUNKS IN MISLABELLED WORDS
      in mislabelled words in all words error_rate
chunk                                             
לא                   2,150       59,845       3.6%
אל                   2,075      180,307       1.2%
לא_                  1,687       25,858       6.5%
לי                   1,659       46,373       3.6%
ול                   1,500       38,211       3.9%
_אל                  1,374      131,786       1.0%
וא                   1,332       52,921       2.5%
לה                   1,236       38,353       3.2%
יה                   1,096       31,825       3.4%
ות                     978       24,086       4.1%



10 MOST FREQUENT CHUNKS IN JUDEO-ARABIC WORDS  (share_HE: how often the chunk occurs in Hebrew words)
      in JA words share_HE
chunk                     
אל        168,286     6.7%
_אל       125,088     5.1%
לא         47,097    21.3%
וא         41,597    21.4%
לי         34,975    24.6%
פי         32,460    11.2%
מא         32,128    12.6%
אן         31,148     2.0%
אן_        31,131     2.0%
_וא        30,909    17.7%



10 MOST FREQUENT CHUNKS IN HEBREW WORDS  (share_HE: how often the chunk occurs in Hebrew words)
      in HE words share_HE
chunk                     
ים         20,349    83.8%
ים_        20,344    83.8%
ות         17,035    70.7%
ני         14,707    54.3%
לא         12,748    21.3%
יו         12,189    77.1%
בי         12,070    58.2%
אל         12,021     6.7%
רי         11,913    57.2%
ין         11,457    53.0%

10 CHUNKS WEIGHTED MOST STRONGLY TOWARDS JUDEO-ARABIC  (~ marks the edge of the word being labelled)
chunk  weight
  ~אל  -18.58
 ~הו~  -13.72
  הא~  -11.45
קולה~  -11.16
 דלך~  -10.75
  אן~  -10.42
~והו~  -10.05
 ~ואל   -9.12
 ~הי~   -8.58
 עני~   -8.24

10 CHUNKS WEIGHTED MOST STRONGLY TOWARDS HEBREW  (~ marks the edge of the word being labelled)
chunk  weight
   ~ה   11.97
~אלא~   11.14
 אלא~    9.15
 יהו~    8.23
  ים~    7.85
 ~אל~    6.98
~אלה~    6.82
  יו~    6.75
 ~אלה    6.73
 ~אלי    6.66
